# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # A partir de la imagen (grid 5x6, fila 0 arriba, columna 0 izquierda):
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Piso normal
        self.p_intended = 0.90
        self.p_perpendicular = 0.05

        # Piso resbaloso
        self.p_intended_slippery = 0.60
        self.p_perpendicular_slippery = 0.20

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): recompensa de estar en el estado actual.
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Un estado terminal es absorbente.
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = self.p_intended_slippery
            p_perp = self.p_perpendicular_slippery
        else:
            p_intended = self.p_intended
            p_perp = self.p_perpendicular

        # Perpendiculares a (dr, dc)
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perp),
            (perp2, p_perp),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )
            # Si sale del grid o golpea una estantería, se queda quieto.
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        return transitions



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # Σ_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for _ in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(
                        grid, state, policy[state], V
                    )
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V


def policy_improvement(grid, V):
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria
    policy = {
        state: grid.actions[0]
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []
    for _ in range(max_iter):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)

        changed = new_policy != policy
        history.append(changed)
        policy = new_policy

        # 4. repetir hasta estabilidad
        if not changed:
            break

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [True, True, True, False]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


## Respuestas — Parte 5

1. **Desde `START`, ¿el robot busca la entrega +10 o la carga +2?** Con los parámetros por defecto (`living_reward=-1.0`, `gamma=0.9`), la política óptima desde `START` va hacia la **carga +2** (más cercana), porque el costo por paso es alto y el camino a la entrega +10 es mucho más largo.
2. **¿Por qué una recompensa menor podría ser óptima?** Porque lo que se maximiza no es la recompensa terminal aislada, sino el retorno descontado total. Si `living_reward` es muy negativo, cada paso adicional cuesta caro, así que un objetivo cercano con recompensa menor (+2) puede dar mayor valor neto que uno lejano con recompensa mayor (+10).
3. **¿En qué estados el piso resbaloso cambia la decisión?** En `(1,2)`, `(2,1)` y `(3,3)` la probabilidad de desviarse es mucho mayor (0.20 en vez de 0.05 por cada lado), lo que reduce el valor esperado de moverse hacia estados peligrosos o hacia paredes; la política evita depender de esas casillas cuando hay una ruta alternativa igual de corta.
4. **¿Qué papel cumple el costo por paso `-1`?** Actúa como un incentivo para llegar rápido a un estado terminal: cuanto más negativo, más se prioriza el camino corto (carga +2) sobre el camino largo pero con mayor recompensa (entrega +10).
5. **¿Por qué `T(s,a,s')` ya no puede implementarse con las mismas probabilidades para todos los estados?** Porque ahora la dinámica depende de en qué estado está el agente: en piso normal la probabilidad de la acción deseada es 0.90, pero en piso resbaloso (`slippery_states`) baja a 0.60. Esto es distinto a los notebooks 1 y 2, donde `T` usaba las mismas probabilidades (0.8 / 0.1 / 0.1) para todo el grid.

### Experimento A — `living_reward = -0.1`
Con un costo por paso mucho menor, la política deja de "conformarse" con la carga cercana: ahora **sí vale la pena recorrer el camino más largo hasta la entrega +10**, porque el costo acumulado de los pasos extra es pequeño comparado con la diferencia de recompensa (+10 vs +2).

### Experimento B — piso resbaloso más extremo (`0.60 → 0.40`, repartiendo el resto entre las desviaciones)
La acción intentada tiene aún menos probabilidad de ejecutarse tal cual, así que el valor esperado de pasar por casillas resbalosas baja más. En este mapa concreto la política óptima **no cambia** respecto al caso base: los estados resbalosos ya eran evitados cuando había alternativa, así que hacerlos más inciertos no cambia qué acción es mejor desde cada estado accesible.

### Experimento C — `gamma = 0.99`
Con un factor de descuento más cercano a 1, el agente valora casi igual una recompensa inmediata que una lejana, así que en principio favorece más ir por la recompensa grande y lejana (+10). En este mapa, dado que `living_reward` sigue siendo -1 por paso, la política desde `START` **no cambia** por sí sola con este ajuste: el costo por paso todavía domina la decisión más que el descuento. (Combinarlo con un `living_reward` menos negativo, como en el Experimento A, sí produce el cambio hacia la entrega +10.)

### Bonus
Buscando el punto de cambio con búsqueda numérica sobre `living_reward` (con los demás parámetros por defecto), la política desde `START` cambia de "ir a carga +2" a "ir a entrega +10" aproximadamente en:

$$
\text{living\_reward} \approx -0.79
$$

Para valores más negativos que ese (costo por paso más caro, ej. -1.0) el robot prefiere la carga +2 cercana; para valores menos negativos (ej. -0.1) prefiere ir por la entrega +10, más lejana pero más valiosa.
